# CSV文件读取 `CsvHelper`

## 成员
- `std::ifstream _ifs`：文件输入流，用于读取CSV文件内容
- `char _buffer[1024]`：行数据缓冲区，存储当前读取的行内容
- `std::string _item_splitter`：字段分隔符字符串
- `std::unordered_map<std::string, int32_t> _fields_map`：字段名到列索引的映射表，用于按字段名查找数据
- `std::vector<std::string> _current_cells`：当前行的字段值列表，存储解析后的各个字段值

## 方法
- **核心属性与构造函数**
  - 构造函数: `CsvReader(const char* item_splitter = ",")`
  - 获取字段（列）总数: `inline uint32_t col_count()`
  - 获取所有字段名称（逗号分隔）: `const char* fields() const`
- **文件加载与迭代**
  - 从文件加载并解析表头: `bool load_from_file(const char* filename)`
  - 读取并解析下一行数据: `bool next_row()`
- **按列索引获取数据**
  - 获取32位有符号整数: `int32_t get_int32(int32_t col)`
  - 获取32位无符号整数: `uint32_t get_uint32(int32_t col)`
  - 获取64位有符号整数: `int64_t get_int64(int32_t col)`
  - 获取64位无符号整数: `uint64_t get_uint64(int32_t col)`
  - 获取双精度浮点数: `double get_double(int32_t col)`
  - 获取字符串: `const char* get_string(int32_t col)`
- **按字段名获取数据**
  - 获取32位有符号整数: `int32_t get_int32(const char* field)`
  - 获取32位无符号整数: `uint32_t get_uint32(const char* field)`
  - 获取64位有符号整数: `int64_t get_int64(const char* field)`
  - 获取64位无符号整数: `uint64_t get_uint64(const char* field)`
  - 获取双精度浮点数: `double get_double(const char* field)`
  - 获取字符串: `const char* get_string(const char* field)`

# 日志管理器 `WTSLogger`
可以通过配置文件构造多个日志器
- 层次
  - 静态日志器 `root`：默认输出目标
  - 其余静态日志器
  - `dyn_pattern`
    - 动态构建日志器模板
- 配置
  - 静态日志器
    - `async`：日志消息是否立即处理，如果不是则被放入一个队列，由一个专门的后台线程负责（降低对主线程的影响）
    - `level`：该日志器只处理 `level` 及以上级别的日志
    - `sinks`：输出配置，可包含多个配置，对于每个配置
      - `filename`：日志文件路径
      - `pattern`：日志格式
      - `level`
      - `type`：输出种类
        - `basic_file_sink`：输出到一个文件中
        - `daily_file_sink`：每天输出到一个新的文件，例如 `filename_2024-09-18.log`
        - `console_sink`：输出到控制台
        - `ostream_sink`：可以输出到任何流对象，这里实现为输出到 std::cout（也是控制台）
  - 动态构建日志器模板
    - `async`
    - `level`
    - `sinks`
      - `filename`：这里含有占位符，用于实际动态构建时替换
      - `pattern`
      - `level`
      - `type`

## 配置文件例子
```JSON
{
    // root 日志器是必须的，它是所有日志的根节点和默认输出目标。
    // 这里配置了一个复杂的 root 日志器，演示了多目标输出 (Multi-Sinks)。
    "root": {
        "async": true,      // 推荐：对根日志器使用异步模式，以最小化对主线程性能的影响。
        "level": "info",    // 日志器级别 (第一道关卡): root 日志器只处理 info 及以上级别的日志。
        "sinks": [
            {
                // 第一个输出目标：每日滚动的日志文件，记录所有通过第一道关卡的日志。
                "type": "daily_file_sink",
                "filename": "logs/wondertrader.log", // 日志文件名，每天会自动生成如 wondertrader_2024-09-18.log 的文件
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] [%n] %v" // 日志格式: [时间戳] [级别] [日志器名] 日志内容
            },
            {
                // 第二个输出目标：彩色控制台输出，但有更严格的级别过滤。
                "type": "console_sink",
                "level": "warn", // Sink 级别 (第二道关卡): 只有 warn 及以上级别的日志才会显示在控制台。
                "pattern": "[%H:%M:%S.%e] [%^%l%$] %v" // %^...%$ 用于给日志级别着色
            },
            {
                // 第三个输出目标：演示 ostream_sink，同样输出到控制台。
                // 它与 console_sink 类似，但更通用，可以输出到任何 ostream。
                "type": "ostream_sink",
                "level": "fatal", // Sink 级别 (第二道关卡): 只有最严重的 fatal 级别日志才会通过这个 sink 输出。
                "pattern": "!!! FATAL !!! [%Y-%m-%d %H:%M:%S.%e] %v"
            }
        ]
    },

    // 为交易模块定义的静态分类日志器。
    "trading": {
        "async": true,
        "level": "debug", // 设置为 debug 级别，以捕获最详细的交易执行信息供复盘使用。
        "sinks": [
            {
                "type": "daily_file_sink",
                "filename": "logs/trading.log",
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
            }
        ]
    },

    // 为风控模块定义的静态分类日志器。
    "risk_control": {
        "async": false, // 设置为同步模式，确保风控相关的日志（特别是错误和警告）被立即写入磁盘，防止程序崩溃时信息丢失。
        "level": "info",
        "sinks": [
            {
                "type": "basic_file_sink",
                "filename": "logs/risk.log",
                "truncate": false, // 不清空文件，持续追加日志。
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
            }
        ]
    },

    // 为数据下载/行情接收模块定义的日志器。
    "data_retriever": {
        "async": true,
        "level": "info",
        "sinks": [
            {
                "type": "basic_file_sink",
                "filename": "logs/data.log",
                "truncate": true, // 每次程序启动时清空日志文件，适用于调试数据连接问题，只关心当次运行的日志。
                "pattern": "[%Y-%m-%d %H:%M:%S.%e] %v"
            }
        ]
    },

    // dyn_pattern 是一个特殊的顶级键，用于定义动态日志的“模板”。
    // 这里定义的不是具体的日志器，而是供程序在运行时按需创建日志器的蓝图。
    "dyn_pattern": {
        // 模板1：用于为每个策略实例创建独立的日志文件。
        "strategy_template": {
            "async": true,
            "level": "debug",
            "sinks": [
                {
                    "type": "daily_file_sink",
                    // 核心: filename 中的 "%s" 是一个占位符。
                    // 在运行时，它会被 log_dyn 函数传入的具体分类名 (如 "MyStrategy_rb2410") 替换。
                    // 这样每个策略实例就会有自己的日志文件，例如：logs/strategies/MyStrategy_rb2410.log
                    "filename": "logs/strategies/%s.log",
                    "pattern": "[%Y-%m-%d %H:%M:%S.%e] [%l] %v"
                }
            ]
        },

        // 模板2：用于追踪单个订单的生命周期，方便精细化调试。
        "order_trace_template": {
            "async": false, // 同步写入，确保订单状态的实时记录。
            "level": "debug",
            "sinks": [
                {
                    "type": "basic_file_sink",
                    // %s 会被替换为订单的本地ID (localid)。
                    // 例如：logs/orders/10001.log
                    "filename": "logs/orders/%s.log",
                    "truncate": true, // 每次追踪（可以理解为程序重启后）都生成一个全新的日志文件。
                    "pattern": "[%H:%M:%S.%e] %v"
                }
            ]
        }
    }
}
```

## 成员以及日志管理器存储位置
- **状态与控制标志**
    - `static bool m_bInited;`：初始化标志位，标记日志系统是否已经通过 init() 函数成功初始化
    - `static bool m_bTpInited;`：异步线程池初始化标志位
      - 当配置文件中任何一个日志器被配置为异步模式时，就需要一个后台线程池来处理日志写入队列
    - `static bool m_bStopped;`：日志系统停止标志位，当被设为 `true` 时，所有日志输出调用都将被忽略
- **核心处理组件**
    - `static ILogHandler* m_logHandler;`：自定义日志处理器指针
      - 可以通过 `init()` 传入，或者 `registerHandler()` 传入一个实现了 `ILogHandler` 接口的对象
      - 之后每一条日志被传入日志处理器，也会被传递给该处理器
    - `static WTSLogLevel m_logLevel;`：全局日志级别过滤器，作为日志的第一道关卡，低于此级别的日志消息会被直接丢弃
    - `static SpdLoggerPtr m_rootLogger;`：根日志器 root 的智能指针，作为所有未指定分类日志的默认输出目标
    - `static thread_local char m_buffer[MAX_LOG_BUF_SIZE];`：线程本地日志缓冲区，每个线程独享一个2KB的缓冲区用于格式化日志消息
- **动态日志管理**
    - `static LogPatterns* m_mapPatterns;`：动态日志模板映射表
      - `LogPatterns` 是 `WTSHashMap<std::string>` 的别名
      - 以模板名称为键，存储从配置文件 `dyn_pattern` 中解析出的 `WTSVariant` 配置对象
    - `static std::set<std::string> m_setDynLoggers;`：动态日志器名称集合
      - 用于追踪所有通过模板在运行时动态创建的日志器的名称

`WTSLogger` 并不直接持有除 `m_rootLogger`（根日志器 root）外的所有日志器实例
- 无论是从配置文件中解析出的静态日志器，还是运行时动态创建的日志器
- 都会通过 `spdlog::register_logger()` 函数注册并存储在 `spdlog` 库内部的一个 *全局静态的哈希表（注册表）* 中
- 当需要使用某个日志器时，`WTSLogger` 会通过 `spdlog::get(logger_name)` 按名称从这个全局注册表中获取

## 方法
- **日志系统管理接口**
    - ***初始化日志系统***
        ```cpp
        /* @param propFile 配置文件路径或配置内容字符串，默认为"logcfg.json"
        * @param isFile 指示propFile参数的类型，true表示文件路径，false表示配置内容
        * @param handler 可选的自定义日志处理器，用于接收所有日志消息
        */
        void WTSLogger::init(const char* propFile /* = "logcfg.json" */, bool isFile /* = true */, ILogHandler* handler /* = NULL */)
        ```
    - ***注册自定义日志处理器***：`static void registerHandler(ILogHandler* handler = NULL) {m_logHandler = handler;}`
    - ***停止日志系统***：`static void stop()`
    - ***释放所有动态创建的日志器***：`static void freeAllDynLoggers()`
- **原始日志输出接口**
    - ***输出原始日志消息到根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
        void WTSLogger::log_raw(WTSLogLevel ll, const char* message)
        ```
    - ***输出原始日志消息到对应静态日志器、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@param catName 日志器名称，用于获取对应的日志器
        * @param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
       void WTSLogger::log_raw_by_cat(const char* catName, WTSLogLevel ll, const char* message)
        ```
    - ***输出原始日志消息到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @param patttern 动态日志模式名称，用于查找对应的配置模板
        * @param catName 日志分类名称，将作为动态创建的日志器名称
        * @param ll 日志级别，用于级别过滤和选择输出方法
        * @param message 已格式化的日志消息内容，不再进行格式化处理
        */
        void WTSLogger::log_dyn_raw(const char* patttern, const char* catName, WTSLogLevel ll, const char* message)
        ```
- **格式化日志输出接口**
    - ***输出调试级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @tparam Args 可变参数模板类型
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        * 例如：WTSLogger::debug("用户{}登录，时间:{}", userId, timestamp);
        */
        template<typename... Args>
        static void debug(const char* format, const Args& ...args)
        ```
    - ***输出信息级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...info...`
    - ***输出警告级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...warn...`
    - ***输出错误级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...error...`
    - ***输出致命错误级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...fatal...`
    - ***输出指定级别的格式化日志到根日志器 root、自定义日志处理器 m_logHandler***：`...log(WTSLogLevel ll, const char* format, const Args& ...args)`
    - ***输出指定级别的格式化日志到指定静态日志器、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /*@tparam Args 可变参数模板类型
        * @param catName 日志分类名称
        * @param ll 日志级别
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        */
        template<typename... Args>
        static void log_by_cat(const char* catName, WTSLogLevel ll, const char* format, const Args& ...args)
        ```
    - ***输出指定级别的格式化日志（且前缀带 "[级别]"）到指定静态日志器、自定义日志处理器 m_logHandler*** `...log_by_cat_prefix...`
    - ***输出指定级别的格式化日志到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***
        ```cpp
        /* @tparam Args 可变参数模板类型
        * @param patttern 日志模式名称
        * @param catName 日志分类名称
        * @param ll 日志级别
        * @param format 格式化字符串（支持fmt格式）
        * @param args 格式化参数
        */
        template<typename... Args>
        static void log_dyn(const char* patttern, const char* catName, WTSLogLevel ll, const char* format, const Args& ...args)
        ```
    - ***输出指定级别的格式化日志（且前缀带 "[级别]"）到对应动态日志器（如果没有则先创建）、根日志器 root、自定义日志处理器 m_logHandler***：`...log_dyn_prefix...`
- **内部实现方法**
    - ***输出调试级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***
        ```cpp
        /* @param logger 目标日志器指针，可以为NULL
        * @param message 已格式化的日志消息内容
        */
        void WTSLogger::debug_imp(SpdLoggerPtr logger, const char* message)
        {
            // 如果指定的日志器存在，输出调试信息到该日志器
            if (logger)
                logger->debug(message);

            // 如果指定的日志器不是根日志器，同时输出到根日志器
            if (logger != m_rootLogger)
                m_rootLogger->debug(message);

            // 如果存在自定义日志处理器，调用其处理方法
            if (m_logHandler)
                m_logHandler->handleLogAppend(LL_DEBUG, message);
        }
        ```
    - ***输出信息级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...info_imp...`
    - ***输出警告级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...warn_imp...`
    - ***输出错误级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...error_imp...`
    - ***输出致命级别日志到指定日志器、根日志器、自定义日志处理器 m_logHandler***：`...fatal_imp...`
    - ***初始化指定分类的日志器***：`static void initLogger(const char* catName, WTSVariant* cfgLogger)`
    - ***获取指定名称的日志器***：`static SpdLoggerPtr getLogger(const char* logger, const char* pattern = "")`
    - ***在控制台打印消息（未初始化时使用）***：`static void print_message(const char* buffer)`

# 基础数据管理器 `WTSBaseDataMgr`
管理交易系统运行的基础信息，包括：合约信息、品种信息、交易时间、节假日等。

继承了 `IBaseDataMgr`，参考 [Includes/note.ipynb/数据管理接口层/基础数据管理接口 IBaseDataMgr](../Includes/note.ipynb)

## 成员
- `TradingDayTplMap m_mapTradingDay`：交易日模板映射表，存储各地区的交易日历模板
  - typedef wt_hashmap\<std::string, `TradingDayTpl`\>	TradingDayTplMap;
    - uint32_t _cur_tdate：当前交易日日期，格式为YYYYMMDD
	- `HolidaySet` _holidays：节假日集合，存储所有节假日日期
    	- typedef wt_hashset\<uint32_t\> HolidaySet;
  - 本质上是 **map<节假日模板ID，(当前交易日期，set(节假日ID))>**
  - 节假日模板ID即本质上是地区ID，因为节假日本质上是按照地区区分的
- `SessionCodeMap m_mapSessionCode`：时段代码映射表，维护时段与品种的对应关系
  - typedef wt_hashmap\<std::string, `CodeSet`\> SessionCodeMap;
    - typedef fastest_hashset\<std::string\> CodeSet;
  - 本质上是 **map<时段ID，set(品种ID)>**
- `WTSExchgContract* m_mapExchgContract`：按交易所组织的合约信息映射表
  - 本质上是 **map<交易所ID，map\*\<合约ID，合约信息WTSContractInfo\*\>\>**
- `WTSSessionMap* m_mapSessions`：交易时段信息映射表
  - 本质上是 **map<品种ID, 品种交易时段信息WTSSessionInfo\*\>**
- `WTSCommodityMap* m_mapCommodities`：商品品种信息映射表
  - 本质上是 **map<品种ID, 品种信息WTSCommodityInfo\*\>**
- `WTSContractMap* m_mapContracts`：合约信息映射表，支持同名合约的多版本管理
  - 本质上是 **map<合约ID, array(合约信息WTSContractInfo\*\)\>**
  - 因为一个合约可能在多个交易所都有

## 方法

### 信息加载接口

#### 加载交易时段配置 loadSessions
```cpp
/* @param filename 交易时段配置文件路径
 * @return bool 加载成功返回true，失败返回false
 */
bool WTSBaseDataMgr::loadSessions(const char* filename)
```

加载交易时段配置文件（JSON）并解析到交易时段信息映射表 `m_mapSessions: *WTSSessionMap`
- 本质上是 `map<ID, 交易时段信息 WTSSessionInfo*>`
- 配置文件（JSON）举例
	```JSON
	{
		"STK_A": {					// 交易时间模板ID
			"name": "A股标准交易",	// 显示名称
			"offset": 0,			// 时间偏移量，单位为分钟。主要用于处理夜盘等跨自然日的交易时段
			"auction": {			// 单个集合竞价时段
				"from": 91500,		// 开始时间，格式为 HHMMSS
				"to": 92500			// 结束时间
			},
			"sections": [			// 连续交易的时间段
				{
					"from": 93000,
					"to": 113000
				},
				{
					"from": 130000,
					"to": 150000
				}
			]
		},
		"COMM_DAY": {
			"name": "商品期货日盘",
			"offset": 0,
			"auction": {
				"from": 85500,
				"to": 90000
			},
			"sections": [
				{
					"from": 90000,
					"to": 101500
				},
				{
					"from": 103000,
					"to": 113000
				},
				{
					"from": 133000,
					"to": 150000
				}
			]
		},
		"COMM_NIGHT": {
			"name": "商品期货夜盘",
			"offset": -180,
			"auction": {
				"from": 205500,
				"to": 210000
			},
			"sections": [
				{
					"from": 210000,
					"to": 230000
				},
				{
					"from": 90000,
					"to": 101500
				},
				{
					"from": 103000,
					"to": 113000
				},
				{
					"from": 133000,
					"to": 150000
				}
			]
		}
	}
	```
- 配置项说明
  - ID：标识符（如"DAY"、"NIGHT"）
  - name：时段显示名称
  - offset：时区偏移量（小时）
  - auction/auctions：集合竞价时间段
  - sections：连续交易时间段数组

#### 加载商品品种配置 loadCommodities
```cpp
/* @param filename 商品品种配置文件路径
 * @return bool 加载成功返回true，失败返回false
 */
bool WTSBaseDataMgr::loadCommodities(const char* filename)
```

加载商品品种配置文件（JSON）并解析到
- 商品品种信息映射表 ` m_mapCommodities: WTSCommodityMap*`
  - 本质上是 `map<品种ID, 品种信息WTSCommodityInfo*>`
- 时段代码映射表 `m_mapSessionCode: SessionCodeMap`
  - 本质上是 `map<交易时间模板ID，set(品种ID)>`

配置文件（JSON）举例
```JSON
{
    "SHFE": {                           // 交易所代码
        "au": {                         // 品种代码
            "name": "黄金",             // 品种的名称
            "session": "COMM_NIGHT",    // 交易时间模板ID，参考 loadSessions，必须与其中ID相同
            "holiday": "CHINA",         // 节假日模板ID，参考 loadHolidays，必须与其中ID相同
            "pricetick": 0.05,          // 最小价格变动单位。这是该品种价格的最小跳动值
            "volscale": 1000,           // 合约乘数或数量单位。例如，黄金期货的volscale为1000，表示每手合约代表1000克黄金
            "category": 1,              // 合约类别。对应于 WTSTypes.h 中的 ContractCategory 枚举
            "covermode": 1,             // 平仓模式。对应于 WTSTypes.h 中的 CoverMode 枚举
            "pricemode": 0,             // 价格模式。对应于 WTSTypes.h 中的 PriceMode 枚举
            "trademode": 0,             // 交易模式。对应于 WTSTypes.h 中的 TradingMode 枚举
            "lotstick": 1.0,            // 最小下单数量单位。例如，股票通常是100股的整数倍，这里可以设为100.0；数字货币可能允许小数下单，如0.00001。
            "minlots": 1.0              // 最小下单数量。例如，股票的最小下单量是100股
        },
        "rb": {
            "name": "螺纹钢",
            "session": "COMM_NIGHT",
            "holiday": "CHINA",
            "pricetick": 1.0,
            "volscale": 10,
            "category": 1,
            "covermode": 1,
            "pricemode": 0,
            "trademode": 0,
            "lotstick": 1.0,
            "minlots": 1.0
        }
    },
    "DCE": {
        "m": {
            "name": "豆粕",
            "session": "COMM_NIGHT",
            "holiday": "CHINA",
            "pricetick": 1.0,
            "volscale": 10,
            "category": 2,
            "covermode": 1,
            "pricemode": 0,
            "trademode": 0,
            "lotstick": 1.0,
            "minlots": 1.0
        }
    },
    "SSE": {
        "600036": {
            "name": "招商银行",
            "session": "STK_A",
            "holiday": "CHINA",
            "pricetick": 0.01,
            "volscale": 1,
            "category": 0,
            "covermode": 3,
            "pricemode": 1,
            "trademode": 2,
            "lotstick": 100.0,
            "minlots": 100.0
        }
    },
    "BINANCE": {
        "BTCUSDT": {
            "name": "比特币/USDT",
            "session": "CRYPTO_247",
            "holiday": "ALL",
            "pricetick": 0.01,
            "volscale": 1,
            "category": 20,
            "covermode": 3,
            "pricemode": 0,
            "trademode": 0,
            "lotstick": 0.00001,
            "minlots": 0.0001
        }
    }
}
```

#### 加载合约信息配置 loadContracts
```cpp
/* @param filename 商品品种配置文件路径
 * @return bool 加载成功返回true，失败返回false
 */
bool WTSBaseDataMgr::loadContracts(const char* filename)
```

加载合约配置文件（JSON）并解析到
- 商品品种信息映射表 ` m_mapCommodities: WTSCommodityMap*`
  - 本质上是 `map<品种ID, 品种信息WTSCommodityInfo*>`
- 时段代码映射表 `m_mapSessionCode: SessionCodeMap`
  - 本质上是 `map<交易时间模板ID，set(品种ID)>`
- 按交易所组织的合约信息映射表 `m_mapExchgContract: WTSExchgContract*`
  - 本质上是 `map<交易所ID，map*<合约ID，合约信息WTSContractInfo*>>`
- 合约信息映射表 `m_mapContracts: WTSContractMap*`
  - 本质上是 `map<合约ID, array(合约信息WTSContractInfo*)>`，一个合约可能在多个交易所都有

配置文件（JSON）举例
```cpp
{
    "SHFE": {                           // 交易所代码
        "au2412": {                     // 合约代码
            "name": "黄金2412",         // 合约名称
            "exchg": "SHFE",            // 合约所属的交易所代码
            // 标准模式，使用 product 字段（m_mapCommodities 有对应的品种信息则直接从中提取，参考 loadCommodities）
            "product": "au",            // 合约所属的品种代码
            "maxmarketqty": 500,        // 最大市价单下单量
            "maxlimitqty": 1000,        // 最大限价单下单量
            "minmarketqty": 1,          // 最小市价单下单量
            "minlimitqty": 1,           // 最小限价单下单量
            "opendate": 20231218,       // 合约的上市日期
            "expiredate": 20241216,     // 合约的到期或交割日期。对于股票等无到期日的品种，可以设为一个未来的日期，如99991231。
            "longmarginratio": 0.08,    // 多头保证金率
            "shortmarginratio": 0.08    // 空头保证金率
        },
        "rb2501": {
            "name": "螺纹钢2501",
            "exchg": "SHFE",
            "product": "rb",
            "maxmarketqty": 1000,
            "maxlimitqty": 2000,
            "minmarketqty": 1,
            "minlimitqty": 1,
            "opendate": 20240119,
            "expiredate": 20250115,
            "longmarginratio": 0.09,
            "shortmarginratio": 0.09
        }
    },
    "CZCE": {
        "SR501": {
            "name": "白糖2501",
            "exchg": "CZCE",
            // 兼容模式，使用 rules 字段（m_mapCommodities 没有对应的品种信息）
            // 直接内联品种信息，参考 loadCommodities
            "rules": {
                "session": "COMM_NIGHT",
                "holiday": "CHINA",
                "pricetick": 1,
                "volscale": 10,
                "category": 1,
                "covermode": 1,
                "pricemode": 0,
                "trademode": 0
            },
            "opendate": 20240108,
            "expiredate": 20250113
        }
    }
}
```

#### 加载节假日配置 loadHolidays
```cpp
/* @param filename 商品品种配置文件路径
 * @return bool 加载成功返回true，失败返回false
 */
bool WTSBaseDataMgr::loadHolidays(const char* filename)
```

加载节假日配置文件（JSON）并解析到交易日模板映射表 `m_mapTradingDay: TradingDayTplMap`
- 本质上是 `map<地区ID，(当前交易日期，set(节假日ID))>`

配置文件（JSON）举例
```cpp
{
    "CHINA": [      // 地区ID
        20240101,   // 节假日日期，格式为 YYYYMMDD 
        20240209,
        20240210,
        20240211,
        20240212,
        20240213,
        20240214,
        20240215,
        20240216,
        20240217,
        20240404,
        20240405,
        20240406,
        20240501,
        20240502,
        20240503,
        20240504,
        20240505,
        20240610,
        20240915,
        20240916,
        20240917,
        20241001,
        20241002,
        20241003,
        20241004,
        20241005,
        20241006,
        20241007
    ],
    "US": [
        20240101,
        20240115,
        20240219,
        20240329,
        20240527,
        20240619,
        20240704,
        20240902,
        20241128,
        20241225
    ],
    "HK": [
        20240101,
        20240210,
        20240212,
        20240213,
        20240329,
        20240401,
        20240404,
        20240501,
        20240515,
        20240610,
        20240701,
        20240918,
        20241001,
        20241011,
        20241225,
        20241226
    ],
    "ALL": [    // 空数组，通常用于那些7x24小时不间断交易的品种，例如数字货币，它们没有任何交易假日
    ]
}
```

### 合约、品种、交易时段查询接口

#### 按标准品种ID获取品种信息 getCommodity
```cpp
/**
 * @brief 根据标准品种ID获取商品信息
 * @param exchgpid 标准品种ID，格式为"交易所.品种代码"（如"SHFE.cu"）
 * @return WTSCommodityInfo* 商品信息指针，未找到返回NULL
 * 
 * 使用示例：
 * - "SHFE.cu" : 上海期货交易所的铜
 * - "DCE.m"   : 大连商品交易所的豆粕  
 */
WTSCommodityInfo* WTSBaseDataMgr::getCommodity(const char* exchgpid)
{
	// 直接使用标准品种ID在商品映射表中查找对应的商品信息
	// 由于使用哈希表存储，查找效率为O(1)
	return (WTSCommodityInfo*)m_mapCommodities->get(exchgpid);
}
```

#### 按交易所和品种代码获取品种信息 getCommodity
```cpp
/**
 * @brief 根据交易所代码和品种代码获取商品信息
 * @param exchg 交易所代码（如"SHFE"、"DCE"、"CZCE"等）
 * @param pid 品种代码（如"cu"、"m"、"CF"等）
 * @return WTSCommodityInfo* 商品信息指针，未找到返回NULL
 */
WTSCommodityInfo* WTSBaseDataMgr::getCommodity(const char* exchg, const char* pid)
{
	if (m_mapCommodities == NULL)
		return NULL;
	char key[64] = { 0 };
	
	// 使用fmt库的format_to函数构建标准品种ID
	// 格式："{交易所代码}.{品种代码}"，如"SHFE.cu"
	fmt::format_to(key, "{}.{}", exchg, pid);

	return (WTSCommodityInfo*)m_mapCommodities->get(key);
}
```

#### 获取单个合约信息 getContract
```cpp
/* @param code 合约代码（如"cu2012"、"000001"等）
 * @param exchg 交易所代码，默认为空字符串（搜索所有交易所）
 * @param uDate 查询日期（YYYYMMDD格式），用于检查合约有效期，默认为0（不检查）
 * 
 * @return WTSContractInfo* 合约信息指针，未找到或已过期返回NULL
 */
WTSContractInfo* WTSBaseDataMgr::getContract(const char* code, const char* exchg /* = "" */, uint32_t uDate /* = 0 */)
```

流程：
- 如果没有指定交易所即 `exchg = ""`
  - 根据合约ID `code` 从 m_mapContracts（map<合约ID, array(合约信息WTSContractInfo*)>）中找出对应的所有项
    - 如果没有日期条件即 `uDate = 0`，返回第一个
    - 否则：返回第一个满足 *上市日期 <= uDate <= 到期日期* 的合约信息
- 否则
  - 根据交易所ID `exchg` 和合约ID `code` 从 m_mapExchgContract（map<交易所ID，map*<合约ID，合约信息WTSContractInfo*>>）中找到对应项
    - 如果没有日期条件即 `uDate = 0`，返回
    - 否则：当满足 *上市日期 <= uDate <= 到期日期* 时返回

#### 获取合约列表 getContracts
```cpp
/* @param exchg 交易所代码，默认为空字符串（获取所有交易所的合约）
 * @param uDate 查询日期（YYYYMMDD格式），用于筛选有效合约，默认为0（不筛选）
 * @return WTSArray* 合约信息数组，调用者负责释放内存
 */
WTSArray* WTSBaseDataMgr::getContracts(const char* exchg /* = "" */, uint32_t uDate /* = 0 */)
```
流程：
- 如果没有指定交易所即 `exchg = ""`：遍历 m_mapExchgContract（map<交易所ID，map*<合约ID，合约信息WTSContractInfo*>>）中所有合约
- 否则：遍历其中所有对应从交易所ID `exchg` 的合约
- 剔除条件：`uDate` !=0 且 *上市日期 <= `uDate` <= 到期日期* 
- 返回所有未被剔除的合约

#### 获取合约数量 getContractSize
```cpp
/* @param exchg 交易所代码，默认为空字符串（统计所有交易所）
 * @param uDate 查询日期（YYYYMMDD格式），用于筛选有效合约，默认为0（不筛选）
 * @return uint32_t 符合条件的合约数量
 */
uint32_t  WTSBaseDataMgr::getContractSize(const char* exchg /* = "" */, uint32_t uDate /* = 0 */)
```
与 `getContracts` 的唯一区别是返回未被剔除的合约数量

#### 按ID获取交易时段信息 getSession
```cpp
/* @param sid 时段ID（如"TRADING"、"DAY"、"NIGHT"等）
 * @return WTSSessionInfo* 交易时段信息指针，未找到返回NULL
 * 
 * 常见的时段ID示例：
 * - "DAY"：日盘交易时段
 * - "NIGHT"：夜盘交易时段  
 * - "TRADING"：完整交易时段（包含日盘和夜盘）
 */
WTSSessionInfo* WTSBaseDataMgr::getSession(const char* sid)
{
	return (WTSSessionInfo*)m_mapSessions->get(sid);
}
```

#### 按合约代码获取交易时段信息 getSessionByCode
查询路径：*合约代码 -> 合约信息 -> 品种信息 -> 交易时段信息*

```cpp
/* @param code 合约代码（如"cu2012"、"000001"等）
 * @param exchg 交易所代码，默认为空字符串（搜索所有交易所）
 * @return WTSSessionInfo* 交易时段信息指针，未找到返回NULL
 */
WTSSessionInfo* WTSBaseDataMgr::getSessionByCode(const char* code, const char* exchg /* = "" */)
{
	// 首先根据合约代码获取合约信息
	WTSContractInfo* ct = getContract(code, exchg);
	if (ct == NULL)
		return NULL;
	return ct->getCommInfo()->getSessionInfo();
}
```

#### 获取所有交易时段信息 getAllSessions
```cpp
/* @return WTSArray* 包含所有交易时段信息的数组，调用者负责释放内存 */
WTSArray* WTSBaseDataMgr::getAllSessions()
{
	WTSArray* ay = WTSArray::create();
	
	for (auto it = m_mapSessions->begin(); it != m_mapSessions->end(); it++)
	{
		// 将时段信息添加到数组，第二个参数true表示增加引用计数
		ay->append(it->second, true);
	}
	
	return ay;
}
```

#### 获取使用某时段的品种集合 getSessionComms
```cpp
/**
 * @brief 获取指定时段的所有品种代码集合
 * @param sid 时段ID（如"DAY"、"NIGHT"等）
 * @return CodeSet* 品种代码集合指针，未找到返回NULL
 */
CodeSet* WTSBaseDataMgr::getSessionComms(const char* sid)
{
	// 在时段代码映射表中查找指定的时段ID
	auto it = m_mapSessionCode.find(sid);
	if (it == m_mapSessionCode.end())
		return NULL;

	// 返回该时段对应的品种代码集合
	return (CodeSet*)&it->second;
}
```

### 交易日历与日期计算

#### 判断是否为节假日 isHoliday
```cpp
/**
 * @brief 判断指定日期是否为节假日（包括周六和周末）
 * @param pid 品种ID或节假日模板ID
 * @param uDate 要检查的日期（YYYYMMDD格式）
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 * @return bool 是节假日返回true，否则返回false
 */
bool WTSBaseDataMgr::isHoliday(const char* pid, uint32_t uDate, bool isTpl /* = false */)
```
流程：
- `isTpl = True` 表示 `pid` 就是节假日模板ID，否则 `pid` 是品种ID（需要先获取其节假日模板ID）
- 从 m_mapTradingDay（map<节假日模板ID，(当前交易日期，set(节假日ID))>）中找出对应的所有节假日
- 如果 `uDate` 是周六或周末或在找出的节假日中，返回真

#### 判断是否为交易日 isTradingDate
既不是周六、周末，也不是节假日的日期。
```cpp
/**
 * @brief 判断指定日期是否为交易日
 * @param pid 品种ID或节假日模板ID
 * @param uDate 要检查的日期（YYYYMMDD格式）
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 * @return bool 是交易日返回true，否则返回false
 */
bool WTSBaseDataMgr::isTradingDate(const char* pid, uint32_t uDate, bool isTpl /* = false */)
{
	uint32_t wd = TimeUtils::getWeekDay(uDate);
	if (wd != 0 && wd != 6 && !isHoliday(pid, uDate, isTpl))
	{
		return true; 
	}
	return false;
}
```

#### 获取下一指定交易日数量后的交易日 getNextTDate
找到 `uDate` 之后的第 `days` 个交易日（跳过周六、周末、以及 `pid` 对应的节假日）
- `isTpl = true` 时 `pid` 就是节假日模板ID；否则是品种ID（需要获取其对应的节假日）
```cpp
/* @param pid 品种ID或节假日模板ID
 * @param uDate 起始日期（YYYYMMDD格式）
 * @param days 向前推进的交易日天数，默认为1
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 * @return uint32_t 下一个交易日（YYYYMMDD格式）
 */
uint32_t WTSBaseDataMgr::getNextTDate(const char* pid, uint32_t uDate, int days /* = 1 */, bool isTpl /* = false */)
```

####  获取上一指定交易日数量前的交易日 getPrevDate
找到 `uDate` 之前的第 `days` 个交易日（跳过周六、周末、以及 `pid` 对应的节假日）
- `isTpl = true` 时 `pid` 就是节假日模板ID；否则是品种ID（需要获取其对应的节假日）
```cpp
/* @param pid 品种ID或节假日模板ID
 * @param uDate 起始日期（YYYYMMDD格式）
 * @param days 向后回退的交易日天数，默认为1
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 * @return uint32_t 前一个交易日（YYYYMMDD格式）
 */
uint32_t WTSBaseDataMgr::getPrevTDate(const char* pid, uint32_t uDate, int days /* = 1 */, bool isTpl /* = false */)
```

#### 根据自然时间计算所属交易日 calcTradingDate
流程：
- 如果 `uDate = 0` 则令 `uDate` 和 `uTime` 为当前日期、当前时间
- 如果 `isSession = true` 即 `stdPID` 为交易时段ID，则获取交易时段信息到 sInfo（否则 `stdPID` 为品种ID，需要先获取对应交易时段ID）
- 计算偏移后的时间 `offMin = (uTime + sInfo.m_uOffsetMins) mod 24*60`$ \in \left[ {0,24*60} \right)$
- 获取 sInfo 的总交易时长
  - 如果为 24*60（24小时全天候交易，如数字货币等）
    - 如果是正偏移且 uTime > offMin（发生回绕）：返回 uDate 的下一天
    - 如果是负偏移且 uTime < offMin（发生回绕）：返回 uDate 的前一天
    - 否则：返回 uDate
  - 否则（非24小时交易，须考虑周六周日、节假日都不是交易日）
    - 如果是正偏移且（uTime > offMin（发生回绕） 或 uDate 为周六/周日）：返回 uDate 的下一个交易日
    - 如果是负偏移
      - 如果 uTime < offMin（发生回绕）：返回 uDate 的上一个交易日
      - 如果没有发生回绕并且 uDate 为周六/周日：返回 uDate 的下一个交易日
    - 否则（无偏移）
      - 如果 uDate 为周六/周日：返回 uDate 的下一个交易日
  - 其余情况，返回 uDate
```cpp
/* @param stdPID 标准品种ID或时段ID
 * @param uDate 自然日期（YYYYMMDD格式），0表示使用当前日期和时间
 * @param uTime 时间（HHMM格式），当uDate为0时会自动获取当前时间
 * @param isSession 是否直接使用时段ID，默认为false（使用品种ID）
 * @return uint32_t 对应的交易日（YYYYMMDD格式）
 */
uint32_t WTSBaseDataMgr::calcTradingDate(const char* stdPID, uint32_t uDate, uint32_t uTime, bool isSession /* = false */)
```

##### 交易日和自然日

**自然日**：从 `00:00:00` 到 `23:59:59`

**交易日**：一个逻辑上的时间单元，所有归属于这个单元的交易，都会在同一个结算日进行清算
- 它的起止时间完全由交易所规定，经常跨越多个自然日。
- 例子：一个 2023 年 10 月 27 日交易日，包含以下几个交易时段
  - `2023年10月26日 晚上 21:00 - 23:00`
  - `2023年10月27日 上午 09:00 - 11:30`
  - `2023年10月27日 下午 13:30 - 15:00`
  - 如果是正偏移，则这几个交易时段属于10月27日这个交易日；如果是负偏移，则这几个交易时段属于10月26日

#### 获取当前交易日 getTradingDate
获取 `pid` 对应的节假日模板的当前交易日的偏移 `uOffDate` 后的交易日。
- `isTpl` 表示 `pid` 是地区ID（节假日模板UD）还是品种ID（如果是品种ID，要先查询其对应的地区ID）
- 如果节假日模板中没有当前交易日，使用当前日期
```cpp
/**
 * @brief 获取交易日
 * @param pid 品种ID或节假日模板ID
 * @param uOffDate 偏移日期（YYYYMMDD格式），0表示使用当前日期
 * @param uOffMinute 偏移分钟数（暂未使用）
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 * @return uint32_t 对应的交易日（YYYYMMDD格式）
 */
uint32_t WTSBaseDataMgr::getTradingDate(const char* pid, uint32_t uOffDate /* = 0 */, uint32_t uOffMinute /* = 0 */, bool isTpl /* = false */)
```

#### 设置当前交易日 setTradingDate
设置 `pid` 对应的节假日模板的当前交易日为 `uDate`。
- `isTpl` 表示 `pid` 是地区ID（节假日模板UD）还是品种ID（如果是品种ID，要先查询其对应的地区ID）
```cpp
/**
 * @brief 设置当前交易日
 * @param pid 品种ID或节假日模板ID
 * @param uDate 要设置的交易日（YYYYMMDD格式）
 * @param isTpl 是否直接使用模板ID，默认为false（使用品种ID）
 */
void WTSBaseDataMgr::setTradingDate(const char* pid, uint32_t uDate, bool isTpl /* = false */)
```

#### 获取交易日边界时间（开/收盘） getBoundaryTime
获取交易日 `tDate`（YYYYMMDD）的开始/结束时间（YYYYMMDDHHMM）
- `isSession` 表示 `stdPID` 是地区ID（节假日模板UD）还是品种ID（如果是品种ID，要先查询其对应的地区ID）
- `isStart` 表示获取开盘时间还是收盘时间

注意：
- 无偏移，交易日就是自然日
- 负偏移，交易时段横跨两个自然日，交易日是前一个自然日
- 正偏移，交易时段横跨两个自然日，交易日是后一个自然日
- 该函数的参数 `tDate` 给出的是交易日
```cpp
/**
 * @brief 获取边界时间
 * @param stdPID 标准品种ID或时段ID
 * @param tDate 交易日期（YYYYMMDD格式），0表示使用当前日期
 * @param isSession 是否直接使用时段ID，默认为false（使用品种ID）
 * @param isStart 是否获取开始时间，默认为true（获取开始时间），false为结束时间
 * @return uint64_t 边界时间（YYYYMMDDHHMM格式），获取失败返回0
 */
uint64_t WTSBaseDataMgr::getBoundaryTime(const char* stdPID, uint32_t tDate, bool isSession /* = false */, bool isStart /* = true */)
```

# 数据工厂类 `WTSDataFactory`
提供完整的K线数据处理功能：从最底层的Tick数据到各种时间周期K线数据的转换和更新操作

继承了 `IDataFactory`，参考 [Includes/note.ipynb/数据管理接口层/数据工厂接口 IDataFactory](../Includes/note.ipynb)

## 方法

- **内部实现方法 - Tick数据更新**
  - 使用Tick数据更新1分钟K线: `WTSBarStruct* updateMin1Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick, bool bAlignSec = false)`
  - 使用Tick数据更新5分钟K线: `WTSBarStruct* updateMin5Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick, bool bAlignSec = false)`
  - 使用Tick数据更新日线数据: `WTSBarStruct* updateDayData(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick)`
  - 使用Tick数据更新秒级K线数据: `WTSBarStruct* updateSecData(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick)`
- **内部实现方法 - 基础K线更新**
  - 使用基础K线数据更新1分钟K线（多倍周期）: `WTSBarStruct* updateMin1Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSBarStruct* newBasicBar, bool bAlignSec = false)`
  - 使用5分钟K线数据更新更大周期的K线: `WTSBarStruct* updateMin5Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSBarStruct* newBasicBar, bool bAlignSec = false)`
- **内部实现方法 - K线数据提取**
  - 从基础K线提取多倍1分钟周期的K线数据: `WTSKlineData* extractMin1Data(WTSKlineSlice* baseKline, uint32_t times, WTSSessionInfo* sInfo, bool bIncludeOpen = true, bool bAlignSec = false)`
  - 从5分钟K线提取更大周期的K线数据: `WTSKlineData* extractMin5Data(WTSKlineSlice* baseKline, uint32_t times, WTSSessionInfo* sInfo, bool bIncludeOpen = true, bool bAlignSec = false)`
  - 从日线数据提取多日周期的K线数据: `WTSKlineData* extractDayData(WTSKlineSlice* baseKline, uint32_t times, bool bIncludeOpen = true)`
- **辅助工具方法**
  - 获取指定分钟的前一个分钟时间: `static uint32_t getPrevMinute(uint32_t curMinute, int period = 1)`
- **IDataFactory接口实现 - K线数据处理与转换**
  - 基于Tick数据更新K线数据: `virtual WTSBarStruct* updateKlineData(WTSKlineData* klineData, WTSTickData* tick, WTSSessionInfo* sInfo, bool bAlignSec = false)`
  - 基于基础周期K线数据更新目标K线: `virtual WTSBarStruct* updateKlineData(WTSKlineData* klineData, WTSBarStruct* newBasicBar, WTSSessionInfo* sInfo, bool bAlignSec = false)`
  - 从基础周期K线数据提取目标周期K线: `virtual WTSKlineData* extractKlineData(WTSKlineSlice* baseKline, WTSKlinePeriod period, uint32_t times, WTSSessionInfo* sInfo, bool bIncludeOpen = true, bool bAlignSec = false)`
  - 从Tick数据直接提取秒级K线数据: `virtual WTSKlineData* extractKlineData(WTSTickSlice* ayTicks, uint32_t seconds, WTSSessionInfo* sInfo, bool bUnixTime = false, bool bAlignSec = false)`
  - 合并多个K线数据: `virtual bool mergeKlineData(WTSKlineData* klineData, WTSKlineData* newKline)`

### 内部实现：Tick数据更新

#### 使用Tick数据更新秒级K线数据 updateSecData
注意：
- ***基础周期为秒级的K线数据，每个K线窗口一般不会跨越两个交易时段***
- WT框架中，默认：
  - WTSlineData 为秒级K线时，其成员 m_bUnixTime 为 true，即存储的K线的时间戳为**标准的Unix时间戳（从1970年1月1日以来的秒数）**
  - 否则 m_bUnixTime 为 false，即存储的K线的时间戳为 HHMMSS 格式

流程：
- 获取 `tick` 的发生时间（HHMMSSMMM——>HHMMSS）
- 换算为 `sinfo` 的交易时段内的秒偏移
- 计算在 `klineData` 中对应 K 线的结束时间（HHMMSS）
$$\frac{交易时段内的秒偏移}{K线间隔} \cdot {K线间隔} + K线间隔$$
- 将上面得到的结束时间和 `klineDate` 中最后一根K线的结束时间比较
  - 如果一致
    - 更新 `klineDate` 的最后一根K线的 close、high、low、vol、money、hold、add
  - 否则
    - 创建一个新的K线数据 WTSBarStruct 返回
```cpp
/**
 * @brief 使用Tick数据更新秒级K线数据
 * @param sInfo 交易时段信息，用于秒级时间计算
 * @param klineData 目标秒级K线数据对象
 * @param tick 新的Tick数据，包含价格、成交量等信息
 * @return WTSBarStruct* 返回新创建的秒级K线结构指针，更新现有K线返回NULL
 */
WTSBarStruct* WTSDataFactory::updateSecData(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick)
```

#### 使用Tick数据更新1分钟K线 updateMin1Data
流程：
- 获取 `tick` 的发生日期 uDate 和时间戳 uTime（HHMM）
- 如果 uTime 不在 `sInfo` 的集合竞价时段或任一交易时段，并且 tick 成交量不为 0：用 tick 更新 `klinkDate` 的最后一根K线，然后返回NULL
- 如果 uTime 在某交易时段的最后一分钟：将其交易时段内累积分钟数 uMinute 减 1
- 计算 tick 所在K线窗口的的结束时间 uBarTime（HHMM）：
  - 如果 `bAlignSec = true`（K线窗口不能跨多个交易时段）
	$$\min \left( {所在交易时段之前的时段的累积分钟数 + \frac{uMinute- 所在交易时段之前的时段的累积分钟数}{K线间隔} \cdot {K线间隔} + K线间隔,所在交易时段的结束时间} \right) $$	
  - 否则
  $$\frac{uMinute}{K线间隔} \cdot {K线间隔} + K线间隔$$
- 更新或创建新的K线：
  - 如果 klineData 不存在K线 || uBarTime > klineData 最后一根K线的时间戳 || tick 的交易日期 > 大于klineData 最后一根K线的日期
    - 根据tick创建新K线，并添加到klineData中，返回klineData最后一根K线
  - 否则如果 klineData 存在K线 && uBarTime < klineData 最后一根K线的时间戳
    - 返回 NULL
  - 否则
    - 更新klineData最后一根K线，返回NULL

```cpp
/* @param sInfo 交易时段信息，用于时间计算和交易时段验证
 * @param klineData 目标1分钟K线数据对象
 * @param tick 新的Tick数据，包含价格、成交量等信息
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成的K线结构指针，无新增返回NULL
 */
WTSBarStruct* WTSDataFactory::updateMin1Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick, bool bAlignSec /* = false */)
```

#### 使用Tick数据更新5分钟K线 updateMin5Data
与 *使用Tick数据更新1分钟K线 updateMin1Data* 过程一致
```cpp
/* @param sInfo 交易时段信息，用于时间计算和交易时段验证
 * @param klineData 目标5分钟K线数据对象
 * @param tick 新的Tick数据，包含价格、成交量等信息
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成的K线结构指针，无新增返回NULL
 */
WTSBarStruct* WTSDataFactory::updateMin5Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick, bool bAlignSec /* = false */)
```

#### 使用Tick数据更新日线数据 updateDayData
注意：***日K的基础周期是一天，没有倍数***

流程：
- 如果 klineData 不存在日K || tick 的交易日期 != klineData 最后一根日K的日期
  - 根据tick创建新日K并返回
- 否则
  - 更新klineData最后一根日K，返回NULL
```cpp
/* @param sInfo 交易时段信息（日线更新中主要用于验证）
 * @param klineData 目标日线数据对象
 * @param tick 新的Tick数据，包含价格、成交量等信息
 * @return WTSBarStruct* 返回新创建的日线结构指针，更新现有日线返回NULL
 */
WTSBarStruct* WTSDataFactory::updateDayData(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSTickData* tick)
```

### 内部实现：基础K线更新

#### 使用基础K线数据更新1分钟K线（多倍周期） updateMin1Data
注意：
- 当 `klineData` 的倍数不为一分钟时，可以认为 `sInfo` 中的交易时段以 *倍数 × 1分钟* 划分为多个 ***K线窗口***
- `newBasicBar` 是单个K线数据，其在交易时段中属于某个K线窗口，当其时间未与该K线窗口的结束时间重合时
- `klineData` ***闭合*** 是指其被一个时间处于最后一个K线窗口的结束时间处的K线数据更新过

流程：
- 如果 `klineData` 是一分钟K线，直接将 `newBasicBar` 追加到 klineData 尾部并设置闭合，返回 klineData 最后一根K线
- 提取 newBasicBar 的日期 uDate，及其所在K线窗口的结束时间戳 uBarTime（HHMM）：
  - 如果 `bAlignSec = true`（K线窗口不能跨多个交易时段）
	$$\min \left( {所在交易时段之前的时段的累积分钟数 + \frac{uMinute- 所在交易时段之前的时段的累积分钟数}{K线间隔} \cdot {K线间隔} + K线间隔,所在交易时段的结束时间} \right) $$	
  - 否则
  $$\frac{uMinute}{K线间隔} \cdot {K线间隔} + K线间隔$$
- 更新或创建新的K线：
  - 如果 klineData 不存在K线 || uDate 和最后一根K线的日期不一致 || uBarTime 和最后一根K线的时间戳不一致
    - 将 newBasicBar 追加到 klineData 的最后
  - 否则
    - 更新klineData最后一根K线
- **如果 klineData 最后一根K线的时间 >  newBasicBar 的时间，设置 klineData 未闭合；否则设置闭合**
- 根据是否创建了新K线返回创建的新K线/NULL
```cpp
/* @param sInfo 交易时段信息，用于时间计算和交易时段验证
 * @param klineData 目标1分钟K线数据对象
 * @param newBasicBar 新的基础K线数据（通常是更小周期的K线）
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成或更新的K线结构指针，无更新返回NULL
 */
WTSBarStruct* WTSDataFactory::updateMin1Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSBarStruct* newBasicBar, bool bAlignSec/* = false*/)
```

#### 使用5分钟K线数据更新更大周期的K线 updateMin5Data
与 *使用基础K线数据更新1分钟K线（多倍周期） updateMin1Data* 基本一致
```cpp
/* @param sInfo 交易时段信息，用于时间计算和交易时段验证
 * @param klineData 目标5分钟K线数据对象
 * @param newBasicBar 新的基础K线数据（通常是1分钟K线）
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成或更新的K线结构指针，无更新返回NULL
 */
WTSBarStruct* WTSDataFactory::updateMin5Data(WTSSessionInfo* sInfo, WTSKlineData* klineData, WTSBarStruct* newBasicBar, bool bAlignSec/* = false*/)
```

### 内部实现：K线数据提取

#### 从基础K线提取多倍1分钟周期的K线数据 extractMin1Data
流程：
- 创建一个 ret: WTSKlineData，用于存放最终生成的大周期K线，设置ret的周期为 `times`
- 循环遍历 `baseKline` 中的每根K线（1分钟线）curBar:
  - 计算当前K线在 `sInfo` 的交易时段内的K线窗口的结束时间的日期 uDate 和时间戳 uBarTime（HHMM）
    - `bAlignSec` 决定K线窗口是否不能跨多个交易时段
  - 聚合或创建：创建lastBar，尝试获取ret的最后一个大K线
    - 如果 ret 中没有K线，或者其中最后一条K线的时间戳和 uBarTime 不一致
      - 将curBar复制到lastBar，并设置lastBar的日期和时间戳为uDate和uBarTime
    - 否则
      - 使用curBar更新lastBar
    - 如果创建了新K，追加到ret最后
- 如果 ret 的最后一根大K的日期或时间戳大于 baseKline 最后一根K线的日期或时间戳
  - 如果 !bIncludeOpen（不包含未闭合的大K）：删除 ret 的最后一根大K
  - 否则：设置 ret 为未闭合

```cpp
/* @param baseKline 基础周期K线数据切片（通常是更小周期的K线）
 * @param times 目标K线的分钟倍数（如5表示5分钟K线）
 * @param sInfo 交易时段信息，用于时间计算和验证
 * @param bIncludeOpen 是否包含未闭合的K线，默认为true
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSKlineData* 返回提取后的1分钟K线数据，失败返回NULL
 */
WTSKlineData* WTSDataFactory::extractMin1Data(WTSKlineSlice* baseKline, uint32_t times, WTSSessionInfo* sInfo, bool bIncludeOpen /* = true */, bool bAlignSec /* = false */)
```

#### 从5分钟K线提取更大周期的K线数据 extractMin5Data
与方法 `extractMin1Data` 完全类似，只是生成的大K的基础周期为 5 分钟。
```cpp
/* @param baseKline 基础周期K线数据切片（通常是1分钟K线）
 * @param times 目标K线的5分钟倍数（如3表示15分钟K线）
 * @param sInfo 交易时段信息，用于时间计算和验证
 * @param bIncludeOpen 是否包含未闭合的K线，默认为true
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSKlineData* 返回提取后的5分钟K线数据，失败返回NULL
 */
WTSKlineData* WTSDataFactory::extractMin5Data(WTSKlineSlice* baseKline, uint32_t times, WTSSessionInfo* sInfo, bool bIncludeOpen /* = true */, bool bAlignSec /* = false */)
```

#### 从日线数据提取多日周期的K线数据 extractDayData
流程：
- 创建一个 ret: WTSKlineData，用于存放最终生成的大周期K线，设置ret的周期为 `times`
- for(count=0; `baseKline` 中的每根K线（日线）curBar; count++):
  - 获取日线的日期 uDate
  - 聚合或创建：创建lastBar，尝试获取ret的最后一个大K线
    - 如果 ret 中没有K线，或者count==steplen
      - 创建新K线lastBar并将curBar复制到其，并设置lastBar的日期为uDate、时间戳为0，重置count为0
    - 否则
      - 使用curBar更新lastBar
    - 如果创建了新K，追加到ret最后
```cpp
/* @param baseKline 基础周期K线数据切片（通常是分钟级K线）
 * @param times 目标日线的天数倍数（如5表示5日线，周线等）
 * @param bIncludeOpen 是否包含未闭合的K线，默认为true
 * @return WTSKlineData* 返回提取后的日线数据，失败返回NULL
 */
WTSKlineData* WTSDataFactory::extractDayData(WTSKlineSlice* baseKline, uint32_t times, bool bIncludeOpen /* = true */)
```

### 辅助工具方法

#### 获取指定时间(HHMM)的前若干分钟时间(HHMM) getPrevMinute
```cpp
/**
 * @brief 获取前N分钟的时间
 * @param curMinute 当前时间（HHMM格式）
 * @param period 向前推进的分钟数，默认为1
 * @return uint32_t 前N分钟的时间（HHMM格式）
 */
uint32_t WTSDataFactory::getPrevMinute(uint32_t curMinute, int period /* = 1 */)
{
    // 1. 将 HHMM 格式转换为从午夜0点开始的总分钟数
    int32_t totalMinutes = (curMinute / 100) * 60 + (curMinute % 100);

    // 2. 执行分钟数的减法
    totalMinutes -= period;

    // 3. 处理跨天的情况（结果为负数）
    // C++的取模运算对负数结果依赖于具体实现，所以用循环更安全
    while (totalMinutes < 0)
    {
        totalMinutes += 1440; // 1440 = 24 * 60，加上一天的分钟数
    }

    // 确保结果在一天之内
    totalMinutes %= 1440;

    // 4. 将总分钟数转换回 HHMM 格式
    uint32_t newHour = totalMinutes / 60;
    uint32_t newMinute = totalMinutes % 60;

    return newHour * 100 + newMinute;
}
```

### IDataFactory接口实现：K线数据处理与转换

#### 基于Tick数据更新K线数据 updateKlineData
- 检查
  - `klineData`/`tick`/`sInfo` 是否为 NULL
  - `klineData` 和 `tick` 的合约代码是否一致
  - `tick` 的HHMM时间戳是否在 `sInfo` 的交易时段内
- 根据 `klineData` 的基础周期 m_kpPeriod
  - KP_Tick：调用 updateSecData
  - KP_Minute1：调用 updateMin1Data
  - KP_Minute5：调用 updateMin5Data
  - KP_DAY：调用 updateDayData
```cpp
/**
 * @brief 基于Tick数据更新K线数据（主入口函数）
 * @param klineData 目标K线数据对象，要更新的K线数据容器
 * @param tick 新的Tick数据，包含最新的价格和成交信息
 * @param sInfo 交易时段信息，用于时间验证和计算
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成或更新的K线结构指针，无更新返回NULL
 * 
 * 支持的K线周期：
 * - KP_Tick：秒级K线（实际上是处理Tick聚合）
 * - KP_Minute1：1分钟K线
 * - KP_Minute5：5分钟K线
 * - KP_DAY：日线数据
 */
WTSBarStruct* WTSDataFactory::updateKlineData(WTSKlineData* klineData, WTSTickData* tick, WTSSessionInfo* sInfo, bool bAlignSec/* = false*/)
```

#### 基于基础周期K线数据更新目标K线 updateKlineData
- 检查
  - `klineData`/`tick`/`sInfo` 是否为 NULL
- 根据 `klineData` 的基础周期 m_kpPeriod，调用 updateMin1Data/updateMin5Data
```cpp
/**
 * @brief 基于基础K线数据更新目标K线（重载函数）
 * @param klineData 目标K线数据对象，要更新的K线数据容器
 * @param newBasicBar 新的基础周期K线数据，已经聚合好的K线结构
 * @param sInfo 交易时段信息，用于时间计算和验证
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSBarStruct* 返回新生成或更新的K线结构指针，无更新返回NULL
 * 
 * 该函数是updateKlineData的重载版本，用于基于已有的基础周期K线数据来更新
 * 更大周期的K线数据。这种方式比直接从Tick数据聚合更高效。
 * 
 * 与Tick版本的区别：
 * - 输入数据已经是聚合后的K线结构，包含完整的OHLCV信息
 * - 不需要进行Tick级别的价格聚合，只需要进行时间窗口聚合
 * - 处理效率更高，适合多级K线的批量更新
 * 
 * 支持的周期转换：
 * - 1分钟 -> N分钟（N > 1）
 * - 5分钟 -> N×5分钟
 * 
 * 注意：该重载版本不支持日线和秒级K线的更新，
 * 这些周期需要使用Tick版本的updateKlineData函数。
 */
WTSBarStruct* WTSDataFactory::updateKlineData(WTSKlineData* klineData, WTSBarStruct* newBasicBar, WTSSessionInfo* sInfo, bool bAlignSec/* = false*/)
```

#### 从Tick数据直接提取秒级K线数据 extractKlineData
- 检查
  - `baseKline` 是否为 NULL或无数据
  - `times` <= 1 或 `period` == KP_TICK
- 根据 `period`，调用 extractDayData/extractMin1Data/extractMin5Data
```cpp
/* @param baseKline 基础周期K线数据切片，作为数据源
 * @param period 目标K线周期类型（KP_Minute1/KP_Minute5/KP_DAY等）
 * @param times 周期倍数（如5分钟K线的times为5）
 * @param sInfo 交易时段信息，用于时间计算和验证
 * @param bIncludeOpen 是否包含未闭合的K线，默认为true
 * @param bAlignSec 是否按交易时段对齐，默认为false
 * @return WTSKlineData* 返回提取后的K线数据对象，失败返回NULL
 * 
 * 负责从基础周期的K线数据中提取出指定周期的K线数据。
 * 
 * 支持的周期转换：
 * - 基础周期 -> 日线（任意基础周期都可以转换为日线）
 * - 基础周期 -> 1分钟的倍数（如5分钟、15分钟、30分钟等）
 * - 基础周期 -> 5分钟的倍数（如15分钟、30分钟、60分钟等）
 */
WTSKlineData* WTSDataFactory::extractKlineData(WTSKlineSlice* baseKline, WTSKlinePeriod period, uint32_t times, WTSSessionInfo* sInfo, 
		bool bIncludeOpen /* = true */, bool bAlignSec /* = false */)
```

#### 合并两个K线数据 mergeKlineData
- 检查
  - `klineData`/`newKline` 是否为 NULL
  - `klineData` 和 `newKline` 的合约代码、基础周期、周期倍数是否一致
- 如果 `klineData` 为空，直接交换数据，返回 true
- 将 `newKline` 中时间戳早于 `klineData` 的插入到其前部，晚于的则插入到其后部，其余忽略
```cpp
/* @param klineData 目标K线数据对象（合并结果存储在这里）
 * @param newKline 待合并的K线数据对象
 * @return bool 合并成功返回true，失败返回false
 */
bool WTSDataFactory::mergeKlineData(WTSKlineData* klineData, WTSKlineData* newKline)
```

# 主力合约管理器 `WTSHotMgr`